In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [ ]:
path = '/workspaces/crmprtd/update_sheet/'
df = pd.read_excel(path + '20250918-Metadata-AllRequiredChanges.xlsx', header = 1)   # pandas automatically uses openpyxl
df_del = df[
    df["ISSUE"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.startswith("delete")
]

In [6]:
station_ids = (
    df_del["Station ID"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)
station_ids[0]


1527

In [7]:
# DELETE FROM obs_raw o
# USING meta_history h
# WHERE o.history_id = h.history_id
#   AND h.station_id = 1527

In [31]:
from tqdm import tqdm
import sqlalchemy as sa

# List of station_ids to delete
station_ids_to_delete = station_ids  # or a subset

delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for sid in tqdm(station_ids_to_delete, desc="Deleting stations"):

        # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
        res_flags = conn.execute(
            delete_obs_flags,
            {"station_id": sid}
        )

        # 2️⃣ obs_raw
        res_obs = conn.execute(
            delete_obs,
            {"station_id": sid}
        )

        # 3️⃣ obs_derived_values (depends on meta_history)
        res_obs_dv = conn.execute(
            delete_obs_derived,
            {"station_id": sid}
        )

        # 4️⃣ meta_history
        res_hist = conn.execute(
            delete_history,
            {"station_id": sid}
        )

        # 5️⃣ meta_station
        res_sta = conn.execute(
            delete_station,
            {"station_id": sid}
        )

        tqdm.write(
            f"Station {sid}: "
            f"flags={res_flags.rowcount}, "
            f"obs_raw={res_obs.rowcount}, "
            f"obs_derived={res_obs_dv.rowcount}, "
            f"meta_history={res_hist.rowcount}, "
            f"meta_station={res_sta.rowcount}"
        )

Deleting stations:   0%|          | 0/346 [00:00<?, ?it/s]

Deleting stations:   1%|          | 2/346 [00:00<00:38,  8.87it/s]

Station 1527: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 1528: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0


Deleting stations:   1%|▏         | 5/346 [00:00<00:36,  9.43it/s]

Station 1529: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 1530: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 1531: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0


...

Station 2519: flags=0, obs_raw=436326, obs_derived=26, meta_history=1, meta_station=1


Deleting stations: 100%|█████████▉| 345/346 [40:30<02:24, 144.41s/it]

Station 13123: flags=0, obs_raw=58556, obs_derived=0, meta_history=1, meta_station=1


Deleting stations: 100%|██████████| 346/346 [40:37<00:00,  7.05s/it] 

Station 2491: flags=0, obs_raw=8959, obs_derived=36, meta_history=1, meta_station=1


In [ ]:
SELECT s.station_id, h.*
FROM meta_station s
LEFT JOIN meta_history h ON s.station_id = h.station_id
LEFT JOIN obs_raw o ON h.history_id = o.history_id
WHERE s.station_id = 12961
GROUP BY s.station_id, h.station_name

In [9]:
check_sql = sa.text("""
SELECT s.station_id, h.station_name, count(o.obs_raw_id) AS n_obs
FROM meta_station s
LEFT JOIN meta_history h ON s.station_id = h.station_id
LEFT JOIN obs_raw o ON h.history_id = o.history_id
WHERE s.station_id = ANY(:station_ids)
GROUP BY s.station_id, h.station_name
""")

with engine.connect() as conn:
    df_check = pd.read_sql(check_sql, conn, params={"station_ids": station_ids[0:20]})

df_check

,station_id,station_name,n_obs
0,1527,Chilliwack,12759
1,1528,North Delta,20186
2,1529,Surrey East,19686
3,1530,Richmond South,19797
4,1531,Burnaby South,12291
5,1532,Pitt Meadows,7777
6,1533,Langley Central,19362
7,1534,Maple Ridge,13164
8,1535,Vancouver Airport,7797
9,1536,Abbotsford Airport,7931
